# Imports

In [326]:
import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

# Problem 1

## Part 1

Reading graph from Wikispeed

In [327]:
G = nx.read_edgelist("./Midterm_2026/wikispeedia.edgelist", create_using = nx.DiGraph)

Evaluating hits centrality

In [328]:
hubs, authorities = nx.hits(G)
hits = pd.DataFrame({'Hub' : hubs, 'Authority' : authorities})

In [329]:
print("Hub with the highest value")
hits[hits['Hub'] == hits['Hub'].max()]

Hub with the highest value


,Hub,Authority
Driving_on_the_left_or_right,0.002274,0.0


In [330]:
print("Authority with the highest value")
hits[hits['Authority'] == hits['Authority'].max()]

Authority with the highest value


,Hub,Authority
United_States,0.001829,0.011525


## Part 2

Total number of nodes, N

In [331]:
N = G.number_of_nodes()

Creating initial state probability

In [332]:
pi0 = np.ones(N,) / N

Creating W matrix

In [333]:
def createW(G: nx.Graph, node_list, alpha: float = 0.1):

    N = G.number_of_nodes()
    W = np.zeros((N, N))

    out_degree = np.array([out for node,out in G.out_degree(node_list)])
    
    # If a node's neighborhood is empty, the entire row = 1/N
    W[out_degree == 0] = 1/N
    
    # For all rows with out_degree > 0 (non empty out neighborhoods)
    # base probability is set to alpha / N
    W[out_degree > 0] = alpha/N
    
    # The nodes actually in out-neightborhoods will recieve an additional probability boost
    A = nx.adjacency_matrix(G, node_list)
    W[out_degree > 0] += (A[out_degree > 0].T * (1 - alpha) / out_degree[out_degree > 0]).T
        
    return W


In [334]:
node_list = list(G.nodes())
alpha = 0.1
W = createW(G, node_list, alpha)

Iterating to find page rank

In [335]:
pi = pi0
pi_new = None
tol = 1E-10
max_iter = 100

for n in range(max_iter):
    pi_new = W.T @ pi
    
    if np.linalg.norm(pi_new - pi) <= tol:
        break
    
    pi = pi_new

In [336]:
calculated_pagerank = dict(zip(node_list, pi))
nx_pagerank = nx.pagerank(G, 1 - alpha)

# Problem 2

## Part 1

Creating **connected** Random Geometrix Graph

In [352]:
N = 100
radius = 0.5
seed = 2026
G = nx.random_geometric_graph(N, radius, seed = seed)

## Part 2

In [353]:
p = 0.2
rng = np.random.default_rng(seed = seed)
x0 = rng.binomial(1, p=p, size = N).astype(float)

In [354]:
def next_state(x, edgelist, rng):
    
    x_new = x.copy()
    # randomly select element from list
    gossip_edge = edgelist[rng.integers(len(edgelist))]
    u, v = gossip_edge
    # Compute Average
    avg = 0.5 * (x[u] + x[v])
    x_new[[u, v]] = avg
    
    return x_new

In [355]:
x = x0
x_hist = {}
for n in range(100):
    
    x_hist[n] = x
    x = next_state(x, list(G.edges()), rng)